# grid_100x100 セマンティックブロック層デモ

**前提**: UE Editor で `grid_100x100` を開き **PIE 実行中** にこのノートを実行してください。

## シナリオ

1. 既存床・既存 `block_*` より十分高い位置に **仮床**（6×6 マス）をスポーン
2. 隅 **10×10** マス（gx,gy = 1..10）を走査し **wall / floor / air** を判定
3. ブロック下面を仮床上面 + **0.15 m** の位置に、隙間なく敷き詰め（`sem_block_*`）
4. ラベルごとに色分け（wall=赤, floor=緑, air=青）し `.semantic_layer_registry.json` に保存

- 仮床内（1..6）→ 主に **floor**
- 仮床外（7..10）→ 主に **air**
- デモ用障害物 (4,4) → **wall**（`spawn_demo_wall=False` で無効化可）

ロジック: `grid_env_10k_semantic.py` / `block_semantic_scan.py`

カーネル: `conda activate simworld`

In [ ]:
import importlib
import sys
from pathlib import Path
from typing import Optional

from simworld.communicator.unrealcv import UnrealCV


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_sem_dir = _root / "dev" / "grid_env_10k_semantic"
_g10k_dir = _root / "dev" / "grid_env_10k"
_geh_dir = _root / "dev" / "grid_env_hri"
for p in (_root, _sem_dir, _g10k_dir, _geh_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

ucv: Optional[UnrealCV] = None
print(f"[Paths] root={_root}")
print(f"[Paths] semantic={_sem_dir}")

In [ ]:
import grid_env_10k_semantic as sem

importlib.reload(sem)

ucv, _ = sem.ensure_connection()
if not ucv.client.isconnected():
    raise RuntimeError("UnrealCV not connected — start grid_100x100 PIE first.")
print("OK: UnrealCV connected")

In [ ]:
result = sem.run_semantic_layer_demo(
    ucv,
    spawn_demo_wall=True,
    demo_wall_cell=(4, 4),
    default_block_mode="F",
    cleanup_before=True,
)
counts = sem.summarize_semantics(result.semantics)
print(f"placed={len(result.blocks)} wall/floor/air={counts}")
print(f"registry={result.registry_path}")

In [ ]:
# 再実行前に sem_* Actor を削除する場合
# sem.cleanup_semantic_layer(ucv, result.blocks)